# DuckDB Analytics MCP: Notebook Tutorial

This notebook walks through a real end-to-end validation of the production-grade MCP server.

## 1) Environment setup
We resolve project root, load the package from `src/`, and print the runtime context.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent.resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'Project root: {ROOT}')
print(f'Python: {sys.version.split()[0]}')

Project root: /home/ahmad/AI/Projects/duckdb-analytics-mcp
Python: 3.12.10


## 2) Run deployment readiness check (`doctor`)
This calls the real CLI command and captures JSON output.

In [2]:
doctor_cmd = ['uv', 'run', 'duckdb-analytics-mcp', 'doctor']
result = subprocess.run(doctor_cmd, cwd=ROOT, check=True, capture_output=True, text=True)
doctor_payload = json.loads(result.stdout)
doctor_payload

{'health': {'status': 'ok',
  'server': 'duckdb_analytics_mcp',
  'dataset_dir': '/home/ahmad/AI/Projects/duckdb-analytics-mcp/data',
  'dataset_count': 2,
  'checked_at': '2026-06-12T11:46:06.078650Z'},
 'dataset_preview_count': 2,
 'dataset_total': 2}

## 3) Use the service layer directly
The MCP tool functions call this service internally, so this validates core business logic.

In [3]:
from duckdb_analytics_mcp.config import get_settings
from duckdb_analytics_mcp.service import AnalyticsService

settings = get_settings()
service = AnalyticsService(settings)

health = service.health()
health.model_dump()

{'status': 'ok',
 'server': 'duckdb_analytics_mcp',
 'dataset_dir': '/home/ahmad/AI/Projects/duckdb-analytics-mcp/data',
 'dataset_count': 2,
 'checked_at': datetime.datetime(2026, 6, 12, 11, 46, 6, 458983, tzinfo=datetime.timezone.utc)}

## 4) List available datasets

In [4]:
catalog = service.list_datasets(limit=10, offset=0)
catalog.model_dump()

{'total_count': 2,
 'count': 2,
 'limit': 10,
 'offset': 0,
 'has_more': False,
 'next_offset': None,
 'datasets': [{'name': 'sales_orders.csv',
   'path': '/home/ahmad/AI/Projects/duckdb-analytics-mcp/data/sales_orders.csv',
   'file_format': 'csv',
   'size_bytes': 1636,
   'modified_at': datetime.datetime(2026, 6, 12, 7, 11, 21, 718234, tzinfo=datetime.timezone.utc)},
  {'name': 'support_tickets.jsonl',
   'path': '/home/ahmad/AI/Projects/duckdb-analytics-mcp/data/support_tickets.jsonl',
   'file_format': 'jsonl',
   'size_bytes': 1747,
   'modified_at': datetime.datetime(2026, 6, 12, 7, 11, 21, 721234, tzinfo=datetime.timezone.utc)}]}

## 5) Describe a dataset schema and sample rows

In [5]:
description = service.describe_dataset('sales_orders.csv', sample_rows=5)
description.model_dump()

{'dataset': {'name': 'sales_orders.csv',
  'path': '/home/ahmad/AI/Projects/duckdb-analytics-mcp/data/sales_orders.csv',
  'file_format': 'csv',
  'size_bytes': 1636,
  'modified_at': datetime.datetime(2026, 6, 12, 7, 11, 21, 718234, tzinfo=datetime.timezone.utc)},
 'row_count': 30,
 'columns': [{'name': 'order_id', 'data_type': 'BIGINT', 'nullable': 'YES'},
  {'name': 'order_date', 'data_type': 'DATE', 'nullable': 'YES'},
  {'name': 'region', 'data_type': 'VARCHAR', 'nullable': 'YES'},
  {'name': 'product', 'data_type': 'VARCHAR', 'nullable': 'YES'},
  {'name': 'channel', 'data_type': 'VARCHAR', 'nullable': 'YES'},
  {'name': 'units', 'data_type': 'BIGINT', 'nullable': 'YES'},
  {'name': 'unit_price', 'data_type': 'DOUBLE', 'nullable': 'YES'},
  {'name': 'discount_pct', 'data_type': 'DOUBLE', 'nullable': 'YES'}],
 'sample_rows': [{'order_id': 1001,
   'order_date': datetime.date(2026, 1, 2),
   'region': 'North',
   'product': 'Laptop',
   'channel': 'Online',
   'units': 3,
   'unit_

## 6) Execute an analytics query (read-only SQL)

In [6]:
revenue_query = '''
SELECT
    region,
    ROUND(SUM(units * unit_price * (1 - discount_pct)), 2) AS revenue
FROM source
GROUP BY region
ORDER BY revenue DESC
'''

revenue_result = service.query_dataset(
    dataset_name='sales_orders.csv',
    sql=revenue_query,
    limit=20,
    offset=0,
)
revenue_result.model_dump()

{'dataset': 'sales_orders.csv',
 'sql': 'SELECT\n    region,\n    ROUND(SUM(units * unit_price * (1 - discount_pct)), 2) AS revenue\nFROM source\nGROUP BY region\nORDER BY revenue DESC',
 'total_count': 4,
 'count': 4,
 'limit': 20,
 'offset': 0,
 'has_more': False,
 'next_offset': None,
 'columns': ['region', 'revenue'],
 'rows': [{'region': 'North', 'revenue': 24646.93},
  {'region': 'West', 'revenue': 23259.54},
  {'region': 'South', 'revenue': 18548.94},
  {'region': 'East', 'revenue': 14552.55}]}

## 7) Validate SQL guard rails (expected rejection)

In [7]:
try:
    service.query_dataset(
        dataset_name='sales_orders.csv',
        sql="DELETE FROM source WHERE region = 'North'",
        limit=10,
        offset=0,
    )
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))

SQLValidationError
Only SELECT/WITH read-only queries are allowed


## 8) Conclusion
The server passes health checks, dataset operations, read-only query execution, and write-attempt blocking.